# CompressRoute: Full GPU Training, Distillation & Compression Pipeline

This notebook trains the **BERT-base Teacher** and **DistilBERT Baseline** on the full 10,003-sample Banking77 dataset for 3 epochs, distills a 4-layer student model with hyperparameter ablation, exports all models to **ONNX FP32**, applies **INT8 dynamic quantization**, and evaluates accuracy on the full 3,080-sample test set.

### Quick Setup:
1. Set Runtime: **Runtime** -> **Change runtime type** -> **T4 GPU**.
2. Click **Runtime** -> **Run all** (Ctrl+F9).
3. Expected execution time: **~8-12 minutes** on a free T4 GPU.
4. Expected test accuracy: **~92-93%** for BERT-base Teacher on the 3,080-sample test set.

In [ ]:
# 1. Verify GPU Allocation
!nvidia-smi

In [ ]:
# 2. Clone Repository and Pull Latest Code
import os
if not os.path.exists('Natural-Language-Processing'):
    !git clone https://github.com/Dhanuskarthi/Natural-Language-Processing.git
%cd Natural-Language-Processing
!git checkout main
!git pull origin main

In [ ]:
# 3. Install Training & Export Dependencies
!pip install -q -r requirements-train.txt

## Step 1: Full Teacher & Baseline Fine-Tuning
Trains BERT-base teacher and DistilBERT baseline on the full Banking77 dataset (9,003 train, 1,000 val, 3,080 test) for 3 epochs at learning rate 2e-5 with batch size 32.

In [ ]:
!python -m compress.finetune_teacher --model_type all

## Step 2: Knowledge Distillation & Ablation
Distills the BERT teacher into a compact 4-layer student model across temperature and alpha settings on the full Banking77 dataset, logs the ablation matrix, and preserves the top-performing student checkpoint.

In [ ]:
!python -m compress.distill

## Step 3: ONNX FP32 Export, INT8 Quantization & Variant Evaluation
Exports PyTorch checkpoints to ONNX FP32 with dynamic axes, verifies numerical parity (< 1e-4), applies dynamic INT8 quantization, and evaluates test accuracy and Macro-F1 across all 9 model variants on the full test set.

In [ ]:
!python -m compress.export_onnx
!python -m compress.quantize

## Step 4: Inspect Accuracy & Compression Results
Displays the final measured variant comparison table.

In [ ]:
import pandas as pd
summary_df = pd.read_csv('benchmark/results/variants_summary.csv')
print(summary_df[['model_id', 'family', 'precision', 'size_mb', 'compression_vs_teacher', 'accuracy', 'macro_f1']].to_string(index=False))

## Step 5: Package and Download Results Bundle
Zips all generated model files (ONNX FP32/INT8, PyTorch checkpoints, tokenizers) and evaluation metric files (JSON/CSV) into `compressroute_full_run.zip` and triggers an automatic browser download.

In [ ]:
import os
import zipfile
from google.colab import files

zip_path = 'compressroute_full_run.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root_dir in ['benchmark/results', 'models']:
        for folder, _, filenames in os.walk(root_dir):
            for filename in filenames:
                filepath = os.path.join(folder, filename)
                zf.write(filepath, arcname=filepath)

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'Successfully bundled {zip_path} ({size_mb:.1f} MB)')
files.download(zip_path)